# pipeline Creation

In [1]:
#install pipeline
#!pip install pipeline

In [2]:
#import the required libraries
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt


from sklearn.pipeline import Pipeline# for creating a pipeline
from sklearn.impute import SimpleImputer # used for null value imputation
from sklearn.compose import ColumnTransformer # used to bring all the processes together
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.ensemble import RandomForestClassifier


In [3]:
#load the dataset
data = pd.read_csv("churn_modeling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
# check shape
data.shape

(10000, 14)

In [5]:
#check info
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [6]:
#check clumns
data.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')

In [7]:
# drop irrelevant columns

data.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True)

data[:4]

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0


In [8]:
#seperate target and feature

X = data.drop("Exited", axis=1)
y = data["Exited"]

In [9]:
#seperate train and test sets
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=.30, stratify=y, random_state=42)

## 1.  pipeline for processing categorical columns

In [10]:
#handling categorical columns
cat_pipeline = Pipeline([
    
    ("cat_imputation", SimpleImputer(fill_value='missing',strategy='constant')),
    
    ("one_hot_encodeing", OneHotEncoder(sparse_output=False, drop="first", dtype='int', handle_unknown="ignore"))
])

In [11]:
#handling numeric columns
num_pipeline = Pipeline([
    
    ("num_imputation", SimpleImputer(strategy= "median")),
    ("feature_scaling", StandardScaler())
])

## 2. seperate numeric and categorical columns

In [12]:
#cat columns

cat_colnm = X.select_dtypes(include=["object", "category"]).columns.tolist()

#num columns
num_colnm = X.select_dtypes(include=np.number).columns.tolist()

## Combine all the processing into one

In [13]:
preprocesser  = ColumnTransformer([
    
    ("category", cat_pipeline, cat_colnm),
    ("numeric", num_pipeline, num_colnm)
])

## create a pipeline

In [14]:
Rf_pipe = Pipeline([
    
    ("preprocessing", preprocesser),
    ("estimator", RandomForestClassifier())
])

In [15]:
# train the model
Rf_pipe.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('estimator', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('category', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
Rf_pipe.score(X_train, y_train)

0.9998571428571429

In [18]:
#predict

y_pred = Rf_pipe.predict(X_test)

In [19]:
#model accuracy
accuracy_score(y_test, y_pred)

0.8626666666666667